## **Library Installation**

In [ ]:
# Instalasi Library yang Dibutuhkan
!pip install pandas numpy scikit-learn transformers torch tensorflow keras --quiet
!pip install sastrawi --quiet # Library untuk stemming bahasa Indonesia
!pip install imbalanced-learn --quiet # Untuk implementasi SMOTE

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.1 MB/s eta 0:00:00


## **Import Library**

In [ ]:
# Impor Modul Dasar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import time
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
import nltk

# Impor Modul Deep Learning & Tokenizer
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Bidirectional, Conv1D, GlobalMaxPool1D, Concatenate, Attention
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
from imblearn.over_sampling import SMOTE
from transformers import AutoTokenizer, TFAutoModel

warnings.filterwarnings("ignore")

In [ ]:
try:
    nltk.data.find('corpora/stopwords')
except:
    nltk.download('stopwords')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## **Data Loading**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
# Load Data
df = pd.read_csv('/content/drive/MyDrive/ScrapingGrab/ulasan_grab_scraping.csv')
df.head()

## **EDA**

### Data Information

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

### NaN Handling

In [ ]:
# Hapus nilai NaN
df.dropna(subset=['Review', 'Rating'], inplace=True)
print(f"Total data setelah menghapus NaN: {len(df)}")

Dataset sudah bersih dan siap melakukan Labeling serta Preprocessing

### Sentiment Labeling

In [ ]:
# Jurnal 8: Skor > 3 = Positif (1), Skor <= 3 = Negatif (0) [cite: 165-166]
def labeling(rating):
    # Rating 4 dan 5 (Positif) -> 1
    if rating > 3:
        return 1
    # Rating 1, 2, dan 3 (Negatif) -> 0
    else:
        return 0

df['Sentimen_Label'] = df['Rating'].apply(labeling)
print("\nPelabelan Sentimen Selesai.")

In [ ]:
df.head()

## **Preprocessing**

### Stemmer & Stopwords Indonesia

In [ ]:
# Inisialisasi Stemmer Sastrawi dan Stopword Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()
list_stopwords = set(stopwords.words('indonesian'))

### Preprocessing Function

In [ ]:
def preprocessing_text(text):
    # a. Case Folding & Cleaning
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)

    # b. Tokenizing
    words = text.split()

    # c. Stopword Removal
    words = [word for word in words if word not in list_stopwords]

    # d. Stemming
    words = [stemmer.stem(word) for word in words]

    return ' '.join(words)

### Preprocessing Execution

In [ ]:
# Terapkan fungsi preprocessing ke kolom Review
print("\nMemulai Text Preprocessing...")
df['Review_Clean'] = df['Review'].apply(preprocessing_text)
print("\nMemulai Text Preprocessing...")
print("Preprocessing Selesai.")

In [ ]:
df['Review_Clean']

### Checkpoint

In [ ]:
save_path = '/content/drive/MyDrive/ScrapingGrab/ulasan_grab_processed.csv'

df_save = df[['Review', 'Rating', 'Review_Clean', 'Sentimen_Label']]
df_save.to_csv(save_path, index=False)

print(f"\nData bersih ({len(df_save)} baris) telah disimpan di:")
print(f"{save_path}")

### Class Imbalance Analysist

In [ ]:
df_processed = pd.read_csv("/content/drive/MyDrive/ScrapingGrab/ulasan_grab_processed.csv")

print("\nAnalisis Keseimbangan Kelas:\n")
count_labels = df_processed['Sentimen_Label'].value_counts()
print(f"Jumlah Label 0 (Negatif): {count_labels.get(0, 0)}")
print(f"Jumlah Label 1 (Positif): {count_labels.get(1, 0)}")

# Hitung Persentase
total = len(df_processed)
persentase_negatif = (count_labels.get(0, 0) / total) * 100
persentase_positif = (count_labels.get(1, 0) / total) * 100

print(f"Persentase Negatif: {persentase_negatif:.2f}%")
print(f"Persentase Positif: {persentase_positif:.2f}%")

print("\nContoh data yang sudah bersih:\n")
print(df_processed[['Review_Clean', 'Sentimen_Label']].head())

**Highly Imbalanced Data**

### Train Test Split

In [ ]:
X_text = df_processed['Review_Clean'].astype(str).values
y = df_processed['Sentimen_Label'].values

# Parameter Tokenisasi (Mengacu pada Jurnal 8, max_words=10000, max_sequence=100)
MAX_WORDS = 10000
MAX_LEN = 100
TEST_SIZE = 0.2

# SPLIT DATA (Training dan Testing)
# Split data sebelum SMOTE. SMOTE HANYA diterapkan pada data latih (X_train, y_train).
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print("\nSplit Data Selesai (80% Train, 20% Test)")
print(f"Data latih awal (sebelum SMOTE): {len(y_train)}")
print(f"Data uji: {len(y_test)}")

### Padding & Tokenization

In [ ]:
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
tokenizer.fit_on_texts(X_train_text)

# Konversi teks ke sequence angka
X_train_sequences = tokenizer.texts_to_sequences(X_train_text)
X_test_sequences = tokenizer.texts_to_sequences(X_test_text)

# Padding sequence
X_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"\nUkuran X_train_padded: {X_train_padded.shape}")

### SMOTE (Synthetic Minority Over-sampling Technique)

In [ ]:
# Menangani imbalanced data dengan rasio 3:1 menggunakan teknik SMOTE

# PENERAPAN SMOTE (Hanya pada Data Latih)
print("\n Penerapan SMOTE")
print("Distribusi kelas latih sebelum SMOTE:")
print(pd.Series(y_train).value_counts())

smote = SMOTE(random_state=SEED)
X_train_resampled_padded, y_train_resampled = smote.fit_resample(X_train_padded, y_train)

print("\nDistribusi kelas latih setelah SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

## **Arsitektur IndoBERT-CNN-BiLSTM-Attention**

In [ ]:
# PENGATURAN PARAMETER MODEL
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN_BERT = 100
EMBEDDING_DIM = 768
LSTM_UNITS = 128
DROPOUT_RATE = 0.3
NUM_CLASSES = 1 # Binary Classification (Sigmoid output)
LEARNING_RATE = 1e-5

In [ ]:
# Custom Layer untuk BERT
class TFBertLayer(tf.keras.layers.Layer):
    def __init__(self, model_name, **kwargs):
        super(TFBertLayer, self).__init__(**kwargs)
        self.bert_model = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        outputs = self.bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            training=training
        )
        return outputs.last_hidden_state

    def get_config(self):
        config = super().get_config()
        return config

In [ ]:
# Custom Layer untuk Attention Mechanism
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.attention_dense = Dense(1, activation='tanh')
        self.softmax = tf.keras.layers.Softmax(axis=1)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        # inputs shape: (batch, sequence_length, features)
        attention_weights = self.attention_dense(inputs)  # (batch, sequence_length, 1)
        attention_weights = self.softmax(attention_weights)  # (batch, sequence_length, 1)

        # Weighted sum
        weighted = inputs * attention_weights  # (batch, sequence_length, features)
        attention_output = tf.reduce_sum(weighted, axis=1)  # (batch, features)

        return attention_output

    def get_config(self):
        config = super().get_config()
        return config

In [ ]:

def build_indobert_cnn_bilstm_attention(model_name=MODEL_NAME, max_len=MAX_LEN_BERT, units=LSTM_UNITS, dropout_rate=DROPOUT_RATE):
    """
    Membangun model IndoBERT-CNN-BiLSTM dengan lapisan Attention.
    (Arsitektur Gabungan Jurnal 1, Jurnal 7, dan IndoBERT)
    """
    # A. INPUT LAYER
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

    # B. INDOBERT LAYER (FEATURE EXTRACTION) - DIPERBAIKI DENGAN CUSTOM LAYER
    try:
        bert_layer = TFBertLayer(model_name, name='bert_encoder')
        sequence_output = bert_layer([input_ids, attention_mask], training=False)
    except Exception as e:
        print(f"Error loading {model_name}: {e}. Cek kembali nama model atau koneksi internet.")
        return None

    # C. CNN-BILSTM LAYERS (SEKUENSIAL FEATURE PROCESSING)
    conv = Conv1D(filters=64, kernel_size=3, activation='relu')(sequence_output)
    bilstm = Bidirectional(LSTM(units, return_sequences=True, dropout=dropout_rate))(conv)

    # Attention Layer - Custom Implementation (DIPERBAIKI)
    attention_output = AttentionLayer()(bilstm)

    global_max_pool = GlobalMaxPool1D()(bilstm)

    combined_features = Concatenate()([global_max_pool, attention_output])

    # D. OUTPUT LAYER
    dense = Dense(64, activation='relu')(combined_features)
    output = Dense(NUM_CLASSES, activation='sigmoid')(dense) # Binary output

    # E. KOMPILASI MODEL
    model = Model(inputs=[input_ids, attention_mask], outputs=output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

    return model

In [ ]:
# TOKENIZER INDOBERT
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

# FUNGSI UNTUK MENGUBAH TEKS MENJADI INPUT INDOBERT
def create_bert_input(texts, tokenizer, max_len=MAX_LEN_BERT):
    tokenized = tokenizer(
        list(texts),
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask']
    }

# PREPARASI DATA INPUT INDOBERT FINAL
X_test_bert = create_bert_input(X_test_text, tokenizer_bert, max_len=MAX_LEN_BERT)


## **Training Model**

In [ ]:
# A. Persiapan Class Weights untuk IndoBERT

# 1. Hitung bobot kelas berdasarkan label asli (y_train)
classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weights = dict(zip(classes, class_weights_array))

print("Bobot Kelas untuk IndoBERT (Class Weighting):")
print(class_weights)

In [ ]:
#  B. Pelatihan Model 1 (CNN-BiLSTM Sederhana dengan SMOTE)

print("\nModel 1: CNN-BiLSTM (SMOTE)")

def build_cnn_bilstm_simple(max_words=MAX_WORDS, max_len=MAX_LEN, units=LSTM_UNITS, dropout_rate=DROPOUT_RATE):
    input_layer = Input(shape=(max_len,))
    embedding = Embedding(max_words, 128, input_length=max_len)(input_layer)

    conv = Conv1D(filters=64, kernel_size=3, activation='relu')(embedding)
    bilstm = Bidirectional(LSTM(units, dropout=dropout_rate))(conv)

    dense = Dense(32, activation='relu')(bilstm)
    output = Dense(NUM_CLASSES, activation='sigmoid')(dense)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model_cnn_bilstm = build_cnn_bilstm_simple()

# Pelatihan menggunakan data SMOTE
history_cnn_bilstm = model_cnn_bilstm.fit(
    X_train_resampled_padded, y_train_resampled,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_padded, y_test),
    verbose=1
)

print("Model 1 dilatih. Akurasi Val (CNN-BiLSTM + SMOTE): {:.4f}".format(
    history_cnn_bilstm.history['val_accuracy'][-1]))

In [ ]:
# C. Pelatihan Model 2 (IndoBERT-CNN-BiLSTM-Attention dengan Class Weighting)

print("\nModel 2: IndoBERT-CNN-BiLSTM-Attention (Class Weighting)")

model_indobert = build_indobert_cnn_bilstm_attention()

if model_indobert:
    # Pelatihan menggunakan data teks asli (X_train_text) dan Class Weights
    history_indobert = model_indobert.fit(
        create_bert_input(X_train_text, tokenizer_bert),
        y_train,
        epochs=3,
        batch_size=16,
        class_weight=class_weights,
        validation_data=(X_test_bert, y_test),
        verbose=1
    )
    print("Model 2 dilatih. Akurasi Val (IndoBERT): {:.4f}".format(
        history_indobert.history['val_accuracy'][-1]))
else:
    print("Model 2 gagal dimuat.")